In [17]:
import pandas as pd
import os

# 파일 경로 설정
raw_path = "../data/raw/"

# 파일 목록 확인
files = os.listdir(raw_path)
print("발견된 파일 목록:")
for f in files:
    print(f)

SystemError: <class 'pandas._libs.tslib.__pyx_defaults'> returned a result with an exception set

In [ ]:
import pandas as pd
import os

raw_path = "../data/raw/"
files = {
    "검봉": "gumbong_flux.csv",
    "가리왕": "gariwang_flux.csv",
    "매화": "maehwa_flux.csv"
}

dfs = {}
for name, file in files.items():
    df = pd.read_csv(raw_path + file, encoding="utf-8-sig")
    df["지점"] = name
    dfs[name] = df
    print(f"{name}: {df.shape} / 결측치 비율: {df.isnull().mean().mean():.2%}")

# 합치기
df_all = pd.concat(dfs.values(), ignore_index=True)
print(f"\n전체: {df_all.shape}")
print(df_all.columns.tolist())

In [ ]:
num_cols = [
    "토양수분1", "토양수분2",
    "잠열 플럭스", "현열 플럭스(보정된 온도)",
    "co2 플럭스", "순복사량",
    "기온 10미터", "습도 10미터",
    "강우량(군락 위)", "지중온도1", "지중온도2",
    "지중 열류량1", "지중 열류량2",
    "퀀텀량(군락 위)"
]

df = df.sort_values(["지점", "관측일시"]).reset_index(drop=True)

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[num_cols] = df.groupby("지점")[num_cols].transform(
    lambda x: x.interpolate(method="linear")
)

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df = df[~((df[col] < Q1 - 3*IQR) | (df[col] > Q3 + 3*IQR))]

print(f"전처리 후: {df.shape}")
df.head()

In [ ]:
df.to_csv("../data/processed/flux_processed.csv", index=False, encoding="utf-8-sig")
print("저장 완료")

In [ ]:
import requests
from urllib.parse import quote

API_KEY = "5c9d733a167dc2b234ae7062a28938d853e92953c9b9998a5071373d5b7eb35e"
END_POINT = "https://apis.data.go.kr/1400377/mtweather"

url = f"{END_POINT}/mountListSearch"
params = {
    "serviceKey": quote(API_KEY, safe=''),
    "numOfRows": 100,
    "pageNo": 1,
    "dataType": "JSON"
}

res = requests.get(url, params=params)
print(res.status_code)
print(res.text[:500])

In [ ]:
import xml.etree.ElementTree as ET

# 전체 관측소 목록 가져오기
url = f"{END_POINT}/mountListSearch"
params = {
    "serviceKey": quote(API_KEY, safe=''),
    "numOfRows": 500,
    "pageNo": 1,
    "dataType": "XML"
}

res = requests.get(url, params=params)
root = ET.fromstring(res.text)

# 강원도 관측소 필터링
stations = []
for item in root.iter("item"):
    obsname = item.findtext("obsname", "")
    obsid = item.findtext("obsid", "")
    localarea = item.findtext("localarea", "")
    
    # 강원도 지역코드 또는 이름으로 필터
    if any(k in obsname for k in ["가평", "강릉", "고성", "국립춘천숲체원", "동해", "삼척", "양구", "양양", "영월", "원주", "인제", "정선", "철원", "철원남북산림협력센터", "춘천", "태백", "평창", "홍천", "화천", "횡성" ]):
        stations.append({"obsid": obsid, "obsname": obsname, "localarea": localarea})

import pandas as pd
df_stations = pd.DataFrame(stations)
print(df_stations)

In [ ]:
print(df_stations[df_stations["obsname"].str.contains("가리왕|매화")])

In [ ]:
# 세 지점 실시간 데이터 수집
target_stations = {
    "검봉산": "2916",
    "가리왕산": "2900",
    "매화산": "2038"
}

url = f"{END_POINT}/mountListSearch"
results = []

for name, obsid in target_stations.items():
    params = {
        "serviceKey": quote(API_KEY, safe=''),
        "numOfRows": 1,
        "pageNo": 1,
        "dataType": "XML",
        "obsid": obsid
    }
    res = requests.get(url, params=params)
    root = ET.fromstring(res.text)
    
    for item in root.iter("item"):
        results.append({
            "지점": name,
            "obsid": obsid,
            "기온2m": item.findtext("tm2m"),
            "기온10m": item.findtext("tm10m"),
            "습도2m": item.findtext("hm2m"),
            "습도10m": item.findtext("hm10m"),
            "풍속2m": item.findtext("ws2m"),
            "풍속10m": item.findtext("ws10m"),
            "강수량": item.findtext("rn"),
            "기압": item.findtext("pa"),
        })

df_aws = pd.DataFrame(results)
print(df_aws)

In [ ]:
import requests

AUTH_KEY = "EzBXCbK3SQGwVwmyt6kBfQ"
BASE_URL = "https://apihub.kma.go.kr/api/typ01/url/awsh.php"

# 삼척 인근 지점, 2024년 1월 1일 00시 테스트
params = {
    "tm": "202401010000",
    "stn": "136",  # 삼척 지점번호
    "help": "0",
    "authKey": AUTH_KEY
}

res = requests.get(BASE_URL, params=params)
print(res.status_code)
print(res.text[:500])

In [ ]:
# 강원도 AWS 관측소 목록 조회
stn_url = "https://apihub.kma.go.kr/api/typ01/url/stn_inf.php"
params = {
    "inf": "AWS",
    "stn": "",
    "tm": "202401010000",
    "help": "0",
    "authKey": AUTH_KEY
}

res = requests.get(stn_url, params=params)
lines = res.text.split("\n")

stations = []
for line in lines:
    if line.startswith("#") or line.strip() == "" or "7777" in line:
        continue
    parts = line.split()
    if len(parts) >= 5:
        stn_id = parts[0]
        lat = parts[2]
        lon = parts[3]
        name = parts[4] if len(parts) > 4 else ""
        stations.append({"stn_id": stn_id, "lat": float(lat), "lon": float(lon), "name": name})

df_stn = pd.DataFrame(stations)

# 강원도 위경도 범위로 필터링 (위도 37.0~38.6, 경도 127.5~129.4)
gangwon = df_stn[
    (df_stn["lat"] >= 37.0) & (df_stn["lat"] <= 38.6) &
    (df_stn["lon"] >= 127.5) & (df_stn["lon"] <= 129.4)
]

print(f"강원도 AWS 관측소: {len(gangwon)}개")
print(gangwon[["stn_id", "name", "lat", "lon"]].to_string())

In [ ]:
print(res.text[:1000])

In [ ]:
stations = []
for line in lines:
    if line.startswith("#") or line.strip() == "" or "7777" in line:
        continue
    parts = line.split()
    if len(parts) >= 9:
        try:
            stn_id = parts[0]
            lon = float(parts[1])
            lat = float(parts[2])
            name = parts[8]
            stations.append({"stn_id": stn_id, "lat": lat, "lon": lon, "name": name})
        except:
            continue

df_stn = pd.DataFrame(stations)

# 강원도 위경도 범위
gangwon = df_stn[
    (df_stn["lat"] >= 37.0) & (df_stn["lat"] <= 38.6) &
    (df_stn["lon"] >= 127.5) & (df_stn["lon"] <= 129.4)
].reset_index(drop=True)

print(f"강원도 AWS 관측소: {len(gangwon)}개")
print(gangwon[["stn_id", "name", "lat", "lon"]].to_string())

In [ ]:
import numpy as np
from datetime import datetime, timedelta
import time

# 플럭스 타워 위치
flux_towers = {
    "검봉산": {"lat": 37.23, "lon": 129.28},
    "가리왕산": {"lat": 37.46, "lon": 128.59},
    "매화산": {"lat": 37.62, "lon": 127.86}
}

# 가장 가까운 AWS 지점 찾기
def find_nearest(tower_lat, tower_lon, df_stn):
    df = df_stn.copy()
    df["dist"] = np.sqrt((df["lat"] - tower_lat)**2 + (df["lon"] - tower_lon)**2)
    return df.nsmallest(3, "dist")[["stn_id", "name", "lat", "lon", "dist"]]

for name, coords in flux_towers.items():
    print(f"\n[{name}] 인근 AWS 관측소:")
    print(find_nearest(coords["lat"], coords["lon"], gangwon))

In [ ]:
target_stns = {
    "검봉산": ["529", "310", "878"],
}

# 시간 목록 생성 (2024.01.01 ~ 2026.06.23, 1시간 단위)
start = datetime(2024, 1, 1, 0, 0)
end = datetime(2026, 6, 23, 0, 0)
time_list = []
cur = start
while cur <= end:
    time_list.append(cur.strftime("%Y%m%d%H%M"))
    cur += timedelta(hours=1)

total_calls = sum(len(v) for v in target_stns.values()) * len(time_list)
print(f"총 수집 시간 포인트: {len(time_list)}개")
print(f"예상 API 호출 수: {total_calls}회")

In [ ]:
# 날짜 목록 생성 (일 단위)
start = datetime(2024, 1, 1)
end = datetime(2026, 6, 23)
date_list = []
cur = start
while cur <= end:
    date_list.append(cur.strftime("%Y%m%d"))
    cur += timedelta(days=1)

total_calls = sum(len(v) for v in target_stns.values()) * len(date_list)
print(f"총 날짜 수: {len(date_list)}일")
print(f"예상 API 호출 수: {total_calls}회")

In [ ]:
BASE_URL_DAY = "https://apihub.kma.go.kr/api/typ01/url/sfc_aws_day.php"

params = {
    "tm2": "20240101",
    "obs": "ta_max",
    "stn": "529",
    "disp": "0",
    "help": "0",
    "authKey": AUTH_KEY
}

res = requests.get(BASE_URL_DAY, params=params)
print(res.status_code)
print(res.text[:500])

In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()
retry = Retry(total=5, backoff_factor=1)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)

def safe_get(params, max_retry=5):
    for attempt in range(max_retry):
        try:
            res = session.get(BASE_URL_DAY, params=params, timeout=10)
            return res
        except Exception as e:
            print(f"  재시도 {attempt+1}/{max_retry} - {e}")
            time.sleep(3 * (attempt + 1))
    return None

print("safe_get 정의 완료")

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import os
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from datetime import datetime, timedelta

AUTH_KEY = "EzBXCbK3SQGwVwmyt6kBfQ"
BASE_URL_DAY = "https://apihub.kma.go.kr/api/typ01/url/sfc_aws_day.php"

session = requests.Session()
retry = Retry(total=5, backoff_factor=1)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)

def safe_get(params, max_retry=5):
    for attempt in range(max_retry):
        try:
            res = session.get(BASE_URL_DAY, params=params, timeout=30)
            return res
        except Exception as e:
            wait = 5 * (attempt + 1)
            print(f"  재시도 {attempt+1}/{max_retry} - {wait}초 대기")
            time.sleep(wait)
    return None

obs_list = ["ta_max", "ta_min", "ta_avg", "rn_day", "hm_avg", "ws_avg"]

start = datetime(2024, 1, 1)
end = datetime(2026, 6, 23)
date_list = []
cur = start
while cur <= end:
    date_list.append(cur.strftime("%Y%m%d"))
    cur += timedelta(days=1)

target_stns = {
    "검봉산": ["529", "310", "878"]
}

print(f"설정 완료 / {len(date_list)}일")

In [ ]:
all_data = {}
total_calls = sum(len(v) for v in target_stns.values()) * len(date_list) * len(obs_list)
call_count = 0
start_time = time.time()

for tower, stns in target_stns.items():
    save_path = f"../data/processed/aws_{tower}.csv"
    
    if os.path.exists(save_path):
        existing = pd.read_csv(save_path)
        done_dates = set(existing["date"].astype(str))
        tower_data = existing.to_dict("records")
        print(f"[{tower}] 이어서 시작 ({len(done_dates)}일 완료됨)")
    else:
        done_dates = set()
        tower_data = []
        print(f"[{tower}] 수집 시작...")

    for i, date in enumerate(date_list):
        if date in done_dates:
            call_count += len(stns) * len(obs_list)
            continue

        day_row = {"date": date}

        for obs in obs_list:
            vals = []
            for stn in stns:
                params = {
                    "tm2": date, "obs": obs, "stn": stn,
                    "disp": "0", "help": "0", "authKey": AUTH_KEY
                }
                res = safe_get(params)
                if res:
                    for line in res.text.split("\n"):
                        if line.startswith("#") or "7777" in line or line.strip() == "":
                            continue
                        parts = line.split()
                        if len(parts) >= 6:
                            try:
                                vals.append(float(parts[5]))
                            except:
                                pass
                call_count += 1
                time.sleep(1.5)

            day_row[obs] = np.mean(vals) if vals else np.nan

        tower_data.append(day_row)

        if (i + 1) % 10 == 0:
            pd.DataFrame(tower_data).to_csv(save_path, index=False, encoding="utf-8-sig")
            elapsed = time.time() - start_time
            progress = call_count / total_calls
            eta = (elapsed / progress - elapsed) if progress > 0 else 0
            print(f"  {i+1}/{len(date_list)}일 | 진행률 {progress*100:.1f}% | 남은시간 약 {eta/60:.1f}분")

    all_data[tower] = pd.DataFrame(tower_data)
    all_data[tower].to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"[{tower}] 완료")

In [ ]:
import requests

AUTH_KEY = "EzBXCbK3SQGwVwmyt6kBfQ"
BASE_URL_DAY = "https://apihub.kma.go.kr/api/typ01/url/sfc_aws_day.php"

# ta_avg 직접 테스트
params = {
    "tm2": "20240601",
    "obs": "ta_avg",
    "stn": "529",
    "disp": "0",
    "help": "0",
    "authKey": AUTH_KEY
}

res = requests.get(BASE_URL_DAY, params=params)
print(res.text)

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd

AUTH_KEY = "EzBXCbK3SQGwVwmyt6kBfQ"
url = "https://apihub.kma.go.kr/api/typ02/openApi/AwsMtlyInfoService/getDailyAwsData"

params = {
    "pageNo": 1,
    "numOfRows": 100,
    "dataType": "XML",
    "year": "2024",
    "month": "01",
    "station": "529",  # 원덕(검봉산 인근)
    "authKey": AUTH_KEY
}

res = requests.get(url, params=params)
print(res.status_code)
print(res.text[:500])

In [16]:
from datetime import datetime
import calendar
import time

def get_aws_monthly(year, month, station):
    params = {
        "pageNo": 1,
        "numOfRows": 31,
        "dataType": "XML",
        "year": str(year),
        "month": str(month).zfill(2),
        "station": str(station),
        "authKey": AUTH_KEY
    }
    res = requests.get(url, params=params, timeout=30)
    root = ET.fromstring(res.text)
    
    records = []
    stn_ko = root.findtext(".//stn_ko", "")
    for info in root.iter("info"):
        records.append({
            "year": year,
            "month": month,
            "day": info.findtext("tm", "").strip(),
            "station": station,
            "stn_ko": stn_ko,
            "ta_day": info.findtext("ta_day", "").strip(),
            "ta_max": info.findtext("ta_max", "").strip(),
            "ta_min": info.findtext("ta_min", "").strip(),
            "ws_day": info.findtext("ws_day", "").strip(),
            "rn_day": info.findtext("rn_day", "").strip(),
        })
    return records

# 세 지점 인근 AWS
target_stns = {
    "검봉산": ["529", "310", "878"],
    "가리왕산": ["563", "217", "597"],
    "매화산": ["349", "212", "573"]
}

# 2024.01 ~ 2026.06
periods = []
for y in [2024, 2025, 2026]:
    for m in range(1, 13):
        if y == 2026 and m > 6:
            break
        periods.append((y, m))

all_records = []
total = sum(len(v) for v in target_stns.values()) * len(periods)
count = 0
start_time = time.time()

for tower, stns in target_stns.items():
    for stn in stns:
        for y, m in periods:
            records = get_aws_monthly(y, m, stn)
            for r in records:
                r["tower"] = tower
            all_records.extend(records)
            count += 1
            time.sleep(0.3)
            
            if count % 30 == 0:
                elapsed = time.time() - start_time
                progress = count / total
                eta = (elapsed / progress - elapsed) if progress > 0 else 0
                print(f"{count}/{total} | 진행률 {progress*100:.1f}% | 남은시간 약 {eta/60:.1f}분")

df_aws_new = pd.DataFrame(all_records)
df_aws_new.to_csv("../data/processed/aws_monthly_all.csv", index=False, encoding="utf-8-sig")
print(f"\n완료: {df_aws_new.shape}")

NameError: name 'requests' is not defined

In [15]:
print(df_aws_new.head())
print(f"\n타워별 데이터 수:")
print(df_aws_new.groupby("tower")["ta_day"].count())
print(f"\n결측치:")
print(df_aws_new.isnull().sum())

NameError: name 'df_aws_new' is not defined

In [12]:
# 숫자 변환
num_cols = ["ta_day", "ta_max", "ta_min", "ws_day", "rn_day"]
for col in num_cols:
    df_aws_new[col] = pd.to_numeric(df_aws_new[col], errors="coerce")

# 타워별 월평균 집계
df_aws_monthly = df_aws_new.groupby(["tower", "year", "month"])[num_cols].mean().reset_index()

# 플럭스 데이터 월별 집계
df_flux = pd.read_csv("../data/processed/flux_processed.csv")
df_flux["관측일시"] = pd.to_datetime(df_flux["관측일시"])
df_flux["year"] = df_flux["관측일시"].dt.year
df_flux["month"] = df_flux["관측일시"].dt.month

flux_cols = ["토양수분1", "토양수분2", "잠열 플럭스", "현열 플럭스(보정된 온도)",
             "co2 플럭스", "순복사량", "기온 10미터", "습도 10미터"]

for col in flux_cols:
    df_flux[col] = pd.to_numeric(df_flux[col], errors="coerce")

tower_map = {"검봉": "검봉산", "가리왕": "가리왕산", "매화": "매화산"}
df_flux["tower"] = df_flux["지점"].map(tower_map)

df_flux_monthly = df_flux.groupby(["tower", "year", "month"])[flux_cols].mean().reset_index()

# 병합
df_merged = pd.merge(df_flux_monthly, df_aws_monthly, on=["tower", "year", "month"], how="inner")

print(f"병합 완료: {df_merged.shape}")
print(df_merged.head())

NameError: name 'pd' is not defined

In [13]:
print(f"병합 완료: {df_merged.shape}")
print(df_merged.head())

NameError: name 'df_merged' is not defined

In [ ]:
print(df_merged.shape)
print(df_merged[["tower", "year", "month"]].drop_duplicates().sort_values(["tower", "year", "month"]))

In [ ]:
print(df_flux["지점"].unique())

In [ ]:
# 원본 플럭스 파일 직접 확인
import os
for f in ["gumbong_flux.csv", "gariwang_flux.csv", "maehwa_flux.csv"]:
    path = f"../data/raw/{f}"
    if os.path.exists(path):
        df_tmp = pd.read_csv(path, nrows=2, encoding="utf-8-sig")
        print(f"{f}: {df_tmp.shape}")
    else:
        print(f"{f}: 파일 없음")

In [ ]:
files = {
    "검봉": "../data/raw/gumbong_flux.csv",
    "가리왕": "../data/raw/gariwang_flux.csv",
    "매화": "../data/raw/maehwa_flux.csv"
}

dfs = []
for name, path in files.items():
    df_tmp = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    df_tmp["지점"] = name
    dfs.append(df_tmp)

df_flux_all = pd.concat(dfs, ignore_index=True)
print(f"전체: {df_flux_all.shape}")
print(df_flux_all["지점"].value_counts())

In [11]:
# 관측일시 변환
df_flux_all["관측일시"] = pd.to_datetime(df_flux_all["관측일시"])
df_flux_all["year"] = df_flux_all["관측일시"].dt.year
df_flux_all["month"] = df_flux_all["관측일시"].dt.month

# 핵심 변수 숫자 변환
flux_cols = ["토양수분1", "토양수분2", "잠열 플럭스", "현열 플럭스(보정된 온도)",
             "co2 플럭스", "순복사량", "기온 10미터", "습도 10미터"]

for col in flux_cols:
    df_flux_all[col] = pd.to_numeric(df_flux_all[col], errors="coerce")

# 타워명 매핑
tower_map = {"검봉": "검봉산", "가리왕": "가리왕산", "매화": "매화산"}
df_flux_all["tower"] = df_flux_all["지점"].map(tower_map)

# 월별 집계
df_flux_monthly = df_flux_all.groupby(["tower", "year", "month"])[flux_cols].mean().reset_index()

# AWS와 병합
df_merged = pd.merge(df_flux_monthly, df_aws_monthly, on=["tower", "year", "month"], how="inner")

print(f"병합 완료: {df_merged.shape}")
print(df_merged["tower"].value_counts())

NameError: name 'pd' is not defined

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from urllib.parse import quote

API_KEY = "5c9d733a167dc2b234ae7062a28938d853e92953c9b9998a5071373d5b7eb35e"
END_POINT = "https://apis.data.go.kr/1400377/mtweather"

url = f"{END_POINT}/mountListSearch"
params = {
    "serviceKey": quote(API_KEY, safe=''),
    "numOfRows": 500,
    "pageNo": 1,
    "dataType": "XML"
}

res = requests.get(url, params=params)
root = ET.fromstring(res.text)

stations = []
for item in root.iter("item"):
    obsname = item.findtext("obsname", "")
    obsid = item.findtext("obsid", "")
    localarea = item.findtext("localarea", "")
    stations.append({"stn_id": obsid, "name": obsname, "localarea": localarea})

df_stn = pd.DataFrame(stations)

# 강원도만 (localarea == 10)
gangwon = df_stn[df_stn["localarea"] == "10"].reset_index(drop=True)

# 위경도 추가 (기술문서에서 확인한 값)
# 산악기상 API에는 위경도가 없으니 기상청 AWS 목록에서 가져온 것 사용
print(f"강원도 관측소: {len(gangwon)}개")

In [5]:
import numpy as np

stn_url = "https://apihub.kma.go.kr/api/typ01/url/stn_inf.php"
AUTH_KEY = "EzBXCbK3SQGwVwmyt6kBfQ"

params = {
    "inf": "AWS",
    "stn": "",
    "tm": "202401010000",
    "help": "0",
    "authKey": AUTH_KEY
}

res = requests.get(stn_url, params=params)
lines = res.text.split("\n")

stations = []
for line in lines:
    if line.startswith("#") or line.strip() == "" or "7777" in line:
        continue
    parts = line.split()
    if len(parts) >= 9:
        try:
            stn_id = parts[0]
            lon = float(parts[1])
            lat = float(parts[2])
            name = parts[8]
            stations.append({"stn_id": stn_id, "lat": lat, "lon": lon, "name": name})
        except:
            continue

df_stn = pd.DataFrame(stations)
gangwon = df_stn[
    (df_stn["lat"] >= 37.0) & (df_stn["lat"] <= 38.6) &
    (df_stn["lon"] >= 127.5) & (df_stn["lon"] <= 129.4)
].reset_index(drop=True)

print(f"강원도 AWS 관측소: {len(gangwon)}개")

NameError: name 'requests' is not defined

In [6]:
import rasterio
import numpy as np

with rasterio.open("../data/raw/NDVI_Gangwon_scaled.tif") as src:
    ndvi = src.read(1)
    print(f"크기: {src.width} x {src.height}")
    print(f"좌표계: {src.crs}")
    print(f"범위: {src.bounds}")
    print(f"NDVI 범위: {ndvi.min()} ~ {ndvi.max()}")

크기: 446 x 358
좌표계: EPSG:4326
범위: BoundingBox(left=127.4978882750837, bottom=36.99711497646249, right=129.50113135867022, top=38.605099335036435)
NDVI 범위: nan ~ nan


C:\Users\ssoos\AppData\Local\Temp\ipykernel_16768\3701520185.py:5: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  ndvi = src.read(1)


In [7]:
from rasterio.sample import sample_gen

with rasterio.open("../data/raw/NDVI_Gangwon_scaled.tif") as src:
    # 강원도 AWS 지점 좌표에서 NDVI 추출
    coords = [(row["lon"], row["lat"]) for _, row in gangwon.iterrows()]
    ndvi_values = list(sample_gen(src, coords))
    
gangwon["ndvi_mean"] = [v[0] for v in ndvi_values]

print(gangwon[["stn_id", "name", "lat", "lon", "ndvi_mean"]].head(10))
print(f"\nNDVI 추출 완료: {gangwon['ndvi_mean'].notna().sum()}개 지점")

NameError: name 'gangwon' is not defined

In [8]:
import geopandas as gpd

gdf = gpd.read_file(r"\Users\ssoos\Desktop\Forest_recovery\data\raw\산림입지도\산림입지도.shp", encoding="cp949")
print(f"shape: {gdf.shape}")
print(f"좌표계: {gdf.crs}")
print(gdf.columns.tolist())
print(gdf.head(2))

SystemError: <class 'pandas._libs.tslib.__pyx_defaults'> returned a result with an exception set

In [9]:
# 좌표계 변환 (UTM-K → WGS84)
gdf = gdf.to_crs(epsg=4326)
print(f"좌표계 변환 완료: {gdf.crs}")

# 우리 연구에 필요한 핵심 컬럼만 선택
key_cols = ["SLTP_CD", "TPGRP_TPCD", "SOIL_DRNGE", "LOCTN_GRDN", 
            "ALTTD_CD", "EIGHT_CD", "WASH_CD", "KOFTR_CD", 
            "AVRG_FRAG", "geometry"]

gdf_slim = gdf[key_cols].copy()

# AWS 106개 지점을 GeoDataFrame으로 변환
import geopandas as gpd
from shapely.geometry import Point

gdf_aws = gpd.GeoDataFrame(
    gangwon,
    geometry=[Point(row["lon"], row["lat"]) for _, row in gangwon.iterrows()],
    crs="EPSG:4326"
)

# 공간 조인 (각 AWS 지점이 속한 산림입지 폴리곤 찾기)
gdf_joined = gpd.sjoin(gdf_aws, gdf_slim, how="left", predicate="within")

print(f"공간 조인 완료: {gdf_joined.shape}")
print(gdf_joined[["stn_id", "name", "SLTP_CD", "TPGRP_TPCD", "SOIL_DRNGE", "EIGHT_CD", "KOFTR_CD"]].head(10))

NameError: name 'gdf' is not defined

In [19]:
# 가장 가까운 산림입지 폴리곤 매핑
gdf_joined = gpd.sjoin_nearest(gdf_aws, gdf_slim, how="left", max_distance=0.1)

print(f"공간 조인 완료: {gdf_joined.shape}")
print(gdf_joined[["stn_id", "name", "SLTP_CD", "TPGRP_TPCD", "SOIL_DRNGE", "EIGHT_CD", "KOFTR_CD"]].head(10))

NameError: name 'gpd' is not defined

In [18]:
gdf_result = gdf_joined[[
    "stn_id", "name", "lat", "lon", "ndvi_mean",
    "SLTP_CD",      # 토양형
    "TPGRP_TPCD",   # 지형구분
    "CLZN_CD",      # 기후대
    "PRRCK_LARG",   # 모암대
    "SOIL_DRNGE",   # 토양배수
    "LOCTN_GRDN",   # 입지경사도
    "ALTTD_CD",     # 표고
    "WASH_CD",      # 침식
    "SLANT_TYP",    # 경사형
    "EIGHT_CD",     # 사면방위
    "WIND_EXDGR",   # 바람노출도
    "WTEFF_DGR",    # 풍화정도
    "VLDTY_SLDP",   # 유효토심
    "KOFTR_CD",     # 수종
    "AVRG_FRAG",    # 평균임령
]].copy()

# 코드 → 의미 변환
soil_map = {
    "01":"갈색건조", "02":"갈색약건", "03":"갈색적윤", "04":"갈색약습",
    "05":"적색계갈색건조", "06":"적색계갈색약건", "07":"적색건조", "08":"적색약건",
    "10":"암적색건조", "11":"암적색약건", "12":"암적색적윤",
    "24":"약침식토양", "25":"강침식토양", "27":"미숙토양", "28":"암쇄토양"
}
topo_map = {
    "01":"산정", "02":"산능선", "03":"산복", "04":"산곡",
    "05":"산록", "06":"계곡", "07":"구릉지", "08":"능선", "10":"평지"
}
climate_map = {"1":"온대북부", "2":"온대중부", "3":"온대남부", "4":"난대"}
rock_map = {"1":"화성암", "2":"퇴적암", "3":"변성암"}
drain_map = {"1":"불량", "2":"보통", "3":"양호", "4":"매우양호"}
wash_map = {"1":"없음", "2":"있음", "3":"많음"}
slant_map = {"1":"상승사면", "2":"평행사면", "3":"하강사면"}
wind_map = {"1":"노출", "2":"보통", "3":"보호"}
weather_map = {"01":"상", "02":"중", "03":"하"}
tree_map = {
    "11005":"강원소나무", "11006":"중부소나무", "11017":"낙엽송",
    "11019":"잣나무", "12017":"상수리나무", "12018":"신갈나무",
    "11002":"리기다소나무", "12999":"기타활엽수", "11999":"기타침엽수"
}

gdf_result["토양형"] = gdf_result["SLTP_CD"].map(soil_map)
gdf_result["지형"] = gdf_result["TPGRP_TPCD"].map(topo_map)
gdf_result["기후대"] = gdf_result["CLZN_CD"].astype(str).map(climate_map)
gdf_result["모암대"] = gdf_result["PRRCK_LARG"].astype(str).map(rock_map)
gdf_result["배수"] = gdf_result["SOIL_DRNGE"].astype(str).map(drain_map)
gdf_result["침식"] = gdf_result["WASH_CD"].astype(str).map(wash_map)
gdf_result["경사형"] = gdf_result["SLANT_TYP"].astype(str).map(slant_map)
gdf_result["바람노출"] = gdf_result["WIND_EXDGR"].astype(str).map(wind_map)
gdf_result["풍화정도"] = gdf_result["WTEFF_DGR"].astype(str).map(weather_map)
gdf_result["수종"] = gdf_result["KOFTR_CD"].astype(str).map(tree_map)

gdf_result.to_csv("../data/processed/gangwon_aws_soil.csv", index=False, encoding="utf-8-sig")
print(f"저장 완료: {gdf_result.shape}")
print(gdf_result[["stn_id", "name", "토양형", "지형", "기후대", "배수", "침식", "경사형", "수종"]].head(10))

NameError: name 'gdf_joined' is not defined